# Imports

In [51]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from huggingface_hub import hf_hub_download

from alpaca_eval.metrics.glm_winrate import make_dmatrix_for_model, fit_LogisticRegressionCV
from notebooks.notebook_helpers import load_annotations


# **Instruction difficulty download**

Download instruction difficulties that were calculated before by training GLM with different fixed terms

In [53]:
out = hf_hub_download(repo_id="tatsu-lab/alpaca_eval",
                      filename="instruction_difficulty.csv",
                      repo_type="dataset",
                      force_download=True)

out

'/Users/simon/.cache/huggingface/hub/datasets--tatsu-lab--alpaca_eval/snapshots/2edc6fad8be6b14ea7230aabfd08188da6b8b814/instruction_difficulty.csv'

In [54]:
hf_instruction_difficulty = pd.read_csv(out, index_col=0).squeeze()
hf_instruction_difficulty

index
0      0.000000
1     -0.101360
2     -1.356212
3      0.700087
4     -1.597457
         ...   
800   -0.272119
801    0.437628
802    1.841663
803   -0.010396
804   -0.656699
Name: instruction_difficulty, Length: 805, dtype: float64

# Extract results of evaluating (annotations)

In [55]:
lb = pd.read_csv("../src/alpaca_eval/leaderboards/data_AlpacaEval_2/weighted_alpaca_eval_gpt4_turbo_leaderboard.csv",
                 index_col=0)

lb

,win_rate,standard_error,n_wins,n_wins_base,n_draws,n_total,discrete_win_rate,mode,avg_length,length_controlled_winrate,lc_standard_error
NullModel,76.919792,0.909010,676,129,0,805,83.975155,community,872,86.457807,0.141800
SelfMoA_gemma-2-9b-it-WPO-HB,77.589552,1.231941,640,165,0,805,79.503106,community,3261,78.539281,0.304279
Shopee-SlimMoA-v1,75.614287,1.270627,621,184,0,805,77.142857,community,1994,77.451543,0.430175
blendaxai-gm-l6-vo31,69.110335,1.328074,562,242,1,805,69.875776,community,1809,76.919812,0.572537
gemma-2-9b-it-WPO-HB,77.825032,1.235586,640,163,2,805,79.627329,community,2285,76.725068,0.424260
...,...,...,...,...,...,...,...,...,...,...,...
oasst-sft-pythia-12b,1.790114,0.398558,13,790,2,805,1.739130,verified,726,3.270102,NaN
guanaco-13b,3.469597,0.551861,22,780,3,805,2.919255,verified,1774,3.003787,NaN
guanaco-7b,2.880002,0.520292,21,783,1,805,2.670807,verified,1364,2.871117,NaN
Qwen1.5-1.8B-Chat,3.705557,0.581175,27,774,3,804,3.544776,verified,2673,2.588499,NaN


For every assessed model, extract results of **805** questions answers vs baseline answers

In [56]:
all_df_annotations = load_annotations(lb)
all_df_annotations = all_df_annotations.query("len_2 != 0").dropna(axis=1)

all_df_annotations

,index,generator_1,generator_2,annotator,preference,len_1,len_2,is_longer2,is_longer1,is_same_length,model,position_component
0,0,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.990014,2104,872,False,True,False,NullModel,1
1,1,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.459366,3394,872,False,True,False,NullModel,1
2,2,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.066485,3025,872,False,True,False,NullModel,1
3,3,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.881640,2908,872,False,True,False,NullModel,0
4,4,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.948271,2341,872,False,True,False,NullModel,0
...,...,...,...,...,...,...,...,...,...,...,...,...
169032,800,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000005,5164,2908,False,True,False,baichuan-13b-chat,1
169033,801,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000001,4416,4869,True,False,False,baichuan-13b-chat,1
169034,802,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000064,3070,1308,False,True,False,baichuan-13b-chat,0
169035,803,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000003,3769,2341,False,True,False,baichuan-13b-chat,1


Add instruction difficulties

In [57]:
all_df_annotations['instruction_difficulty'] = all_df_annotations['index'].transform(
    lambda i: hf_instruction_difficulty[i])
all_df_annotations

,index,generator_1,generator_2,annotator,preference,len_1,len_2,is_longer2,is_longer1,is_same_length,model,position_component,instruction_difficulty
0,0,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.990014,2104,872,False,True,False,NullModel,1,0.000000
1,1,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.459366,3394,872,False,True,False,NullModel,1,-0.101360
2,2,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.066485,3025,872,False,True,False,NullModel,1,-1.356212
3,3,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.881640,2908,872,False,True,False,NullModel,0,0.700087
4,4,gpt4_1106_preview,NullModel,weighted_alpaca_eval_gpt4_turbo,0.948271,2341,872,False,True,False,NullModel,0,-1.597457
...,...,...,...,...,...,...,...,...,...,...,...,...,...
169032,800,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000005,5164,2908,False,True,False,baichuan-13b-chat,1,-0.272119
169033,801,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000001,4416,4869,True,False,False,baichuan-13b-chat,1,0.437628
169034,802,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000064,3070,1308,False,True,False,baichuan-13b-chat,0,1.841663
169035,803,gpt4_1106_preview,baichuan-13b-chat,weighted_alpaca_eval_gpt4_turbo,0.000003,3769,2341,False,True,False,baichuan-13b-chat,1,-0.010396


Extract the name of baseline

In [58]:
BASELINE_NAME = all_df_annotations['generator_1'].unique()[0]
BASELINE_NAME

'gpt4_1106_preview'

Drop useless features

In [59]:
all_df_annotations_dropped = all_df_annotations.drop(
    ['model', 'generator_1', 'is_longer1', 'is_longer2', 'is_same_length', 'annotator', 'index'], axis=1)
all_df_annotations_dropped

,generator_2,preference,len_1,len_2,position_component,instruction_difficulty
0,NullModel,0.990014,2104,872,1,0.000000
1,NullModel,0.459366,3394,872,1,-0.101360
2,NullModel,0.066485,3025,872,1,-1.356212
3,NullModel,0.881640,2908,872,0,0.700087
4,NullModel,0.948271,2341,872,0,-1.597457
...,...,...,...,...,...,...
169032,baichuan-13b-chat,0.000005,5164,2908,1,-0.272119
169033,baichuan-13b-chat,0.000001,4416,4869,1,0.437628
169034,baichuan-13b-chat,0.000064,3070,1308,0,1.841663
169035,baichuan-13b-chat,0.000003,3769,2341,1,-0.010396


# Train dataset preparation

Calculate len diff component by formula `diff_component = np.tanh((len_2 - len_1) / std((len_2 - len_1)))` + drop useless components 

In [60]:
train_df = all_df_annotations_dropped.copy()

train_df['len_diff'] = train_df['len_2'] - train_df['len_1']
std_diffs = train_df['len_diff'].std()

train_df['diff_component'] = np.tanh(train_df['len_diff'] / std_diffs)

train_df = train_df.drop(['len_1', 'len_2', 'len_diff'], axis=1)
train_df

,generator_2,preference,position_component,instruction_difficulty,diff_component
0,NullModel,0.990014,1,0.000000,-0.810112
1,NullModel,0.459366,1,-0.101360,-0.980401
2,NullModel,0.066485,1,-1.356212,-0.961855
3,NullModel,0.881640,0,0.700087,-0.952961
4,NullModel,0.948271,0,-1.597457,-0.872683
...,...,...,...,...,...
169032,baichuan-13b-chat,0.000005,1,-0.272119,-0.968305
169033,baichuan-13b-chat,0.000001,1,0.437628,0.392306
169034,baichuan-13b-chat,0.000064,0,1.841663,-0.923505
169035,baichuan-13b-chat,0.000003,1,-0.010396,-0.863440


Prepare test dataset (temporary, **not used right now**)

In [61]:
test_df = train_df.copy()
test_df['diff_component'] = 0

test_df

,generator_2,preference,position_component,instruction_difficulty,diff_component
0,NullModel,0.990014,1,0.000000,0
1,NullModel,0.459366,1,-0.101360,0
2,NullModel,0.066485,1,-1.356212,0
3,NullModel,0.881640,0,0.700087,0
4,NullModel,0.948271,0,-1.597457,0
...,...,...,...,...,...
169032,baichuan-13b-chat,0.000005,1,-0.272119,0
169033,baichuan-13b-chat,0.000001,1,0.437628,0
169034,baichuan-13b-chat,0.000064,0,1.841663,0
169035,baichuan-13b-chat,0.000003,1,-0.010396,0


Extract `patsy` matrices for the logistic regression train

In [62]:
formula = f"C(generator_2, Treatment(reference='{BASELINE_NAME}')) + diff_component + instruction_difficulty + position_component - 1"

df_XY_train, _ = make_dmatrix_for_model(train_df, test_df, formula)
df_XY_train

,"C(generator_2, Treatment(reference='gpt4_1106_preview'))[Conifer-7B-DPO]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Contextual-KTO-Mistral-PairRM]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[FsfairX-Zephyr-Chat-v0.1]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0613-Llama3-70B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0613-Mistral-7B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Llama3-70B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Llama3-8B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Mistral-7B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Qwen2-7B]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[Infinity-Instruct-3M-0625-Yi-1.5-9B]",...,"C(generator_2, Treatment(reference='gpt4_1106_preview'))[xwinlm-7b-v0.1]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[yi-large-preview]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[zephyr-7b-alpha]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[zephyr-7b-alpha-ExPO]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[zephyr-7b-beta]","C(generator_2, Treatment(reference='gpt4_1106_preview'))[zephyr-7b-beta-ExPO]",diff_component,instruction_difficulty,position_component,preference
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.810112,0.000000,1.0,0.990014
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.980401,-0.101360,1.0,0.459366
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.961855,-1.356212,1.0,0.066485
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.952961,0.700087,0.0,0.881640
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.872683,-1.597457,0.0,0.948271
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169032,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.968305,-0.272119,1.0,0.000005
169033,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.392306,0.437628,1.0,0.000001
169034,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.923505,1.841663,0.0,0.000064
169035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.863440,-0.010396,1.0,0.000003


Train logistic regression with the prepared matrix (it's just a model, not a statistical analysis - based on `sklearn`). 
Training features:
- Intercept term is disabled (fit_intercept=False) because the categorical generator variable already encodes model-level offsets (θₘ), and adding a separate intercept would introduce redundancy.
- Logistic regression with **L2 regularization** is used to prevent overfitting and penalize extreme coefficient values, particularly for the length bias term.
- **Cross-entropy** (log loss) is the optimization objective, ensuring the model learns probabilistic preferences accurately.
- Probabilistic labels (preference ∈ [0, 1]) are handled by duplicating each data point: once with label 1 and weight = p, once with label 0 and weight = 1 - p. This allows the model to optimize log loss directly on soft targets.
- Cross-validation (5-fold) is used during training to ensure the robustness of learned coefficients and regularization.

In [63]:
model = fit_LogisticRegressionCV(
    data=df_XY_train,
    col_y_true="preference",
    is_ytrue_proba=True,
    n_splits=5,
    penalty="l2",
    solver="lbfgs",
    fit_intercept=False
)

model

LogisticRegressionCV(cv=GroupKFold(n_splits=5, random_state=None, shuffle=False),
                     fit_intercept=False, random_state=123,
                     scoring=make_scorer(logloss_continuous, greater_is_better=False, response_method='predict_proba'))

Final coefficient for **position bias term**

In [64]:
coef_df = pd.DataFrame({
    "feature": model.feature_names_in_,
    "coef": model.coef_.flatten()
})

coef_df[coef_df['feature'] == 'position_component']

,feature,coef
212,position_component,-0.24325


# Significance check
Since `scikit‑learn’s LogisticRegression` does not support statistical inference on model coefficients, I switched to using `statsmodels` to fit a logistic (logit) regression and obtain **p‑values** for each predictor, omitting both cross‑validation and regularization in that step.

The threshold you choose for deciding whether a p‑value is “significant” is called the significance level

In [65]:
SIGNIFICANCE_LEVEL = 0.05

Change to binominal preference

In [66]:
binominal_train_df = train_df.copy()
binominal_train_df['preference'] = (binominal_train_df['preference'] >= 0.5).astype(int)

binominal_train_df

,generator_2,preference,position_component,instruction_difficulty,diff_component
0,NullModel,1,1,0.000000,-0.810112
1,NullModel,0,1,-0.101360,-0.980401
2,NullModel,0,1,-1.356212,-0.961855
3,NullModel,1,0,0.700087,-0.952961
4,NullModel,1,0,-1.597457,-0.872683
...,...,...,...,...,...
169032,baichuan-13b-chat,0,1,-0.272119,-0.968305
169033,baichuan-13b-chat,0,1,0.437628,0.392306
169034,baichuan-13b-chat,0,0,1.841663,-0.923505
169035,baichuan-13b-chat,0,1,-0.010396,-0.863440


In [67]:
binominal_train_df.describe()

,preference,position_component,instruction_difficulty,diff_component
count,169008.000000,169008.000000,169008.000000,169008.000000
mean,0.209245,0.505515,-0.057163,-0.316035
std,0.406770,0.499971,1.480527,0.513870
min,0.000000,0.000000,-3.052593,-0.999989
25%,0.000000,0.000000,-1.152511,-0.776607
50%,0.000000,1.000000,-0.229853,-0.380631
75%,0.000000,1.000000,0.786168,0.018299
max,1.000000,1.000000,5.811319,1.000000


Train the model with the same `patsy` formula (including target variable)

In [72]:
result = smf.logit(
    formula=f"preference ~ {formula}",
    data=binominal_train_df,
).fit()

result

         Current function value: 0.273098
         Iterations: 35


/Users/simon/PycharmProjects/alpaca_eval/.venv/lib/python3.9/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


We are testing the position bias component (`position_component`) and see that its p-value is below our chosen threshold, indicating that the position bias effect is statistically significant.

In [73]:
summary_df = result.summary2().tables[1]
diff_row = summary_df.loc["position_component"]

diff_row

Coef.      -1.497545e-01
Std.Err.    1.813181e-02
z          -8.259215e+00
P>|z|       1.466332e-16
[0.025     -1.852922e-01
0.975]     -1.142168e-01
Name: position_component, dtype: float64

In [74]:
bool(diff_row['P>|z|'] < SIGNIFICANCE_LEVEL)

True